In [1]:
import pickle, os
import numpy as np
from collections import Counter
from datasets import Dataset
from lowresource_llm_evaluation.LanguageDatasets import LanguageDataset
from transformers import AutoTokenizer


MODELO = "Qwen/Qwen2.5-7B-Instruct" 
TOKENIZER = AutoTokenizer.from_pretrained(MODELO, trust_remote_code=True)

# ---------------------------------------------------------
# 3. Función principal integrada
# ---------------------------------------------------------
def saveTrainTest(
    language,
    modelo_name, # Nombre para la carpeta
    tokenizer,
    thr=0.3,
    compute_length=True, # Por defecto True para auto-ajustar max_length
    concatenate=False,
    max_tokens=1024,
    N_max=20000,
    max_length=None
):
    root_dir = "TrainDatasets"
    os.makedirs(root_dir, exist_ok=True)

    # El nombre del modelo suele tener '/', lo limpiamos para la carpeta
    modelo_folder = modelo_name.split("/")[-1]
    save_dir = f"{root_dir}/{modelo_folder}"
    os.makedirs(save_dir, exist_ok=True)

    # 1. Carga y filtrado inicial
    ds = (
        LanguageDataset(language, filter_language_thr=thr)
        .read_opus(source="NLLB", version=1)
        .filter_by_language(1024, top_k=10)
    )
    ds.get_stats(tokenizer, N_max=N_max)

    # 2. Concatenación (Nuevo método interno)
    if concatenate:
        ds.concatenate(tokenizer, max_tokens=max_tokens)

    # 3. Estadísticas y Auto-ajuste de max_length
    # Si max_length es None, forzamos compute_length para saber el percentil 95
    if compute_length or max_length is None:
        lengths = ds.get_stats(tokenizer, N_max=N_max)
        
        if max_length is None and lengths is not None:
            max_length = int(np.percentile(lengths, 95))
            print(f"🎯 max_length ajustado automáticamente al P95: {max_length}")

    # 4. Split y Tokenización
    # El método split ya devuelve objetos Dataset de HF listos para entrenar
    train, test = ds.split(
        test_size=0.05,
        tokenizer=tokenizer,
        max_length=max_length if max_length else 512
    )

    # 5. Guardar
    save_path = f"{save_dir}/{language}.pkl"
    with open(save_path, "wb") as f:
        pickle.dump((train, test), f, protocol=5)

    print(f"\n💾 Dataset guardado: {save_path} (Train: {len(train)}, Test: {len(test)})")
    return train, test

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [2]:
language = "asturiano"
train, test = saveTrainTest(
    language,
    MODELO,
    TOKENIZER,
    thr=0.3,
    compute_length=True,
    concatenate=True,
    max_tokens=150,
    N_max=200000
)

[INFO] Descargando FastText LID-176 a /usr/local/lib/python3.11/dist-packages/lowresource_llm_evaluation/models/lid.176.ftz ...
[INFO] Modelo FastText descargado correctamente.


Empezando descarga
Descarga completada. 
Procesando las líneas
Dataset cargado

📊 Analizando estadísticas de tokens (N=200000)...
------------------------------------------------
Total líneas en dataset: 935479
Media: 49.14 | Mediana: 43.00
Percentil 95: 104.00 (recomendado para max_length)
Percentil 98: 123.00
Máximo: 191 | Mínimo: 6
Moda (aprox): 30 tokens
------------------------------------------------

🔗 Concatenando por orígenes...
✅ Concatenación finalizada respetando orígenes.

Resumen del dataset para 'asturiano':
Total de líneas: 377545

- opus_NLLB_1: 377545 líneas (100.00%)

Parámetros internos:
  MIN_WORDS: 4
  MAX_WORDS: inf
  MAX_WORD_LEN: 25
  Se anonimiza el texto: True

Códigos de idioma:
  {'tatoeba': 'ast', 'opus': 'ast', 'fasttext': '__label__ast'}

Ejemplos del dataset (primeras 3 líneas):
  • Lleer el testu de la unidá estremando bien les partes narraes de les dialogaes.
Armón acepta acondicionar l’estelleru pa cumplir coles midíes de seguridá esixíes polos traba

Map:   0%|          | 0/358667 [00:00<?, ? examples/s]

Map:   0%|          | 0/18878 [00:00<?, ? examples/s]


💾 Dataset guardado: TrainDatasets/Qwen2.5-7B-Instruct/asturiano.pkl (Train: 358667, Test: 18878)


In [ ]:
language = "aranes"
saveTrainTest(language, MODELO, TOKENIZER, 0.4)

Empezando descarga
Descarga completada. 
Procesando las líneas


: 

: 